# Apple Detector — Kaggle Training (Synthetik-Sweep)

**Workflow:** Das Daten-Dataset wird **einmal** hochgeladen (per `package_for_kaggle.py`).
Für jeden Sweep-Punkt änderst du hier nur `SYNTHETIC_N`, baust das Trainingsset aus
`/kaggle/input` zusammen (benchmark-frei, Symlinks in `/kaggle/working`), trainierst und
evaluierst auf dem **eingefrorenen Benchmark** — für alle Modelle identisch.

> Voraussetzung: Die Repo-Commits müssen nach GitHub gepusht sein (das Notebook klont von dort),
> und das Daten-Dataset muss an dieses Notebook als Input angehängt sein.


## Setup

In [ ]:
!git clone https://github.com/philippsnr/apple-vision.git
%cd apple-vision

In [ ]:
!cd /kaggle/working/apple-vision && git pull && uv sync --quiet

## Konfiguration — ein Sweep-Punkt

Für den Sweep **nur `SYNTHETIC_N`** variieren (z.B. 0 → 100 → 200 → 400 → 800 → 'all').
Reale Basis, Seed und Augmentierung über den gesamten Sweep **konstant** halten,
sonst ist die Accuracy-Kurve nicht vergleichbar.

In [ ]:
import os

# --- Pfade (KAGGLE_INPUT an deinen Dataset-Slug anpassen!) ---
KAGGLE_INPUT = '/kaggle/input/apple-vision-data'
REPO = '/kaggle/working/apple-vision'
WORK = '/kaggle/working'

# --- Sweep-Regler: NUR diese Zahl variieren (0, 100, 200, 400, 800, 'all') ---
SYNTHETIC_N = 200

# --- Reale Basis: über den GESAMTEN Sweep fix halten ---
REAL_SOURCES = {'minneapple': 'all', 'apple_mots': 'all', 'orchard': 'all'}
SEED = 42
VAL_RATIO = 0.1

# --- Augmentierung: über den GESAMTEN Sweep KONSTANT halten (sonst Confound) ---
AUG_FACTOR = 4
AUG_FLAGS = '--aug-brightness 0.3 --aug-contrast 0.3 --aug-saturation 0.2 --aug-hue 0.05 --aug-hflip 0.5'

# --- abgeleitete Pfade ---
TRAIN_ROOT = f'{WORK}/train_set/coco'
BENCH_ROOT = f'{KAGGLE_INPUT}/benchmark/coco'
MANIFEST   = f'{KAGGLE_INPUT}/benchmark_manifest.json'
os.makedirs(f'{WORK}/checkpoints', exist_ok=True)
print('Sweep-Punkt: synthetic =', SYNTHETIC_N)


## Check — Input-Dataset vollständig?

In [ ]:
import os, json

ok = True
for tag in list(REAL_SOURCES) + ['synthetic']:
    ann = f'{KAGGLE_INPUT}/{tag}/coco/annotations/instances_train.json'
    if os.path.exists(ann):
        print(f'{tag:12s} train images: {len(json.load(open(ann))["images"])}')
    else:
        print(f'{tag:12s} FEHLT: {ann}'); ok = False

bench = f'{BENCH_ROOT}/annotations/instances_test.json'
print('benchmark   :', len(json.load(open(bench))['images']) if os.path.exists(bench) else 'FEHLT', 'images')
print('manifest    :', 'OK' if os.path.exists(MANIFEST) else 'FEHLT')
assert ok and os.path.exists(bench) and os.path.exists(MANIFEST), 'Input unvollständig — KAGGLE_INPUT prüfen'


## Trainingsset zusammenbauen (aus /kaggle/input, garantiert benchmark-frei)

In [ ]:
sources = ' '.join(f'--source {KAGGLE_INPUT}/{tag}/coco:{n}' for tag, n in REAL_SOURCES.items())
sources += f' --source {KAGGLE_INPUT}/synthetic/coco:{SYNTHETIC_N}'

compose = (f'cd {REPO} && uv run python scripts/build_training_set.py {sources} '
           f'--benchmark-manifest {MANIFEST} --output {TRAIN_ROOT} '
           f'--val-ratio {VAL_RATIO} --val-exclude synthetic --seed {SEED}')
print(compose)
!{compose}


## Training

In [ ]:
train = (f'cd {REPO} && MPLBACKEND=agg uv run python -m apple_vision.train '
         f'--dataset-root {TRAIN_ROOT} --epochs 30 --batch-size 4 --num-workers 4 '
         f'--early-stop-patience 5 --aug-factor {AUG_FACTOR} {AUG_FLAGS} '
         f'--out-dir {WORK}/checkpoints')
print(train)
!{train}


In [ ]:
from IPython.display import display, Image
display(Image(filename=f'{WORK}/checkpoints/detector_loss.png'))


## Evaluate — eingefrorener Benchmark (für ALLE Modelle identisch)

Ergebnis pro Sweep-Punkt separat gespeichert (`coco_results_synth<N>.json`) — daraus
später die Accuracy-über-Synthetik-Kurve aggregieren.

In [ ]:
evalc = (f'cd {REPO} && uv run python -m apple_vision.evaluate_coco '
         f'--dataset-root {BENCH_ROOT} --val-ann annotations/instances_test.json '
         f'--val-images images/test '
         f'--checkpoint {WORK}/checkpoints/fasterrcnn_resnet50_fpn_apple_best.pth '
         f'--results-json {WORK}/checkpoints/coco_results_synth{SYNTHETIC_N}.json')
print(evalc)
!{evalc}


## Visualize — Predictions auf dem Benchmark

In [ ]:
vis = (f'cd {REPO} && MPLBACKEND=agg uv run python -m apple_vision.visualize_detections '
       f'--checkpoint {WORK}/checkpoints/fasterrcnn_resnet50_fpn_apple_best.pth '
       f'--dataset-root {BENCH_ROOT} --split test --score-threshold 0.5 --n 8 '
       f'--out-dir {WORK}/quickplots/detections')
print(vis)
!{vis}


In [ ]:
from IPython.display import display, Image
from pathlib import Path

for p in sorted(Path(f'{WORK}/quickplots/detections').glob('*.png'))[:4]:
    display(Image(filename=str(p)))
